# Baseline-literature attack grid (FEMNIST) -- Part 1 of 5: Sign-flip @ 10%, Sign-flip @ 20%, Sign-flip @ 30%

**One of 5 independent notebooks** splitting the 12-cell
`RUN_MVP_BASELINE_ATTACK_GRID_FEMNIST.ipynb` (tiny_cnn+FEMNIST, 4
attacks x 10%/20%/30%, 20 rounds each -- the gap-#4
spot-check combo, same attacks/fractions/rounds as the already-
committed tiny_cnn+MNIST grid) so the total runtime can run as 5
parallel Colab/Kaggle sessions instead of one long sequential one.
Each part is fully standalone -- its own clone/install/dataset setup,
no dependency on any other part.

**This part's cells**: Sign-flip @ 10%, Sign-flip @ 20%, Sign-flip @ 30% (3 of the 12 total).


**The other 4 parts** (run them in separate tabs/sessions, any order,
none depend on each other): Part 2 (Gaussian noise @ 10%, Gaussian noise @ 20%, Gaussian noise @ 30%); Part 3 (Scale-to-threshold @ 10%, Scale-to-threshold @ 20%); Part 4 (Scale-to-threshold @ 30%, Low-and-slow @ 10%); Part 5 (Low-and-slow @ 20%, Low-and-slow @ 30%).

See `attacks/baseline_norm_attacks.py` and
`notebooks/_build_baseline_attack_grid_femnist_notebook.py`'s own
title markdown for the full attack citations, the FEMNIST-specific
scope note (this grid deliberately covers tiny_cnn+FEMNIST only, not
small_resnet+MNIST or small_resnet+FEMNIST), and the low-and-slow
epoch-length caveat (20 rounds is not a definitive
answer for that one attack specifically -- still true here if this
part includes it).

**Checkpoints**: identical branch/clone-dir naming to the single
FEMNIST grid notebook -- each cell below keeps its own original grid
index (`0`-`2` of the 12), distinct from
every MNIST-grid branch too. Nothing collides between parts, between
datasets, or with any other notebook in this repo.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


In [ ]:
# 5. Load FEMNIST, writer-disjoint non-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.femnist.cache_dir points at a
# Drive path this run's Drive doesn't have mounted -- overridden here
# to a local Colab cache dir instead, same no-Drive pattern the MNIST
# run uses for its "source" override. flwr-datasets downloads and
# caches FEMNIST (~200MB) to this path over the network, no Drive
# dependency at all.
from fl.data_noniid import prepare_case2

case2_data = prepare_case2(seed=CFG['seed'], cache_dir="/content/femnist_cache/")
print("clients:", len(case2_data['client_train']))
print("server_root writers:", len(case2_data['split'].server_root_writers))
print("unseen_test writers (headline metric source):", len(case2_data['split'].unseen_test_writers))


## Grid cell 1 of 12 (FEMNIST, part 1 of 5): Sign-flip @ 10% malicious (5 of 50 clients)

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 6.5. §6.2 real opening-protocol handler, wired to
# run_training_loop's on_pseudonym_flagged callback (fl/round_loop.py,
# 2026-11-19). This is the SAME run_opening_protocol +
# revoke + residual_anonymity_bits machinery already validated
# standalone in notebooks/traceability_demo/TRACEABILITY_DEMO.ipynb
# (both against the local pure-Python backend and, per FIX.md, the
# real rbcl backend on Colab) -- nothing new is introduced here, this
# cell just triggers it from REAL per-round evidence instead of a
# hand-built test case, using the SAME committee (key_holder_shares,
# pk_shares) the ring cell above just built for escrow.
#
# None of these key holders is ever one of this run's clients (two
# disjoint simulated roles), so no holder ever recuses as
# the accused party (fl/trace.py::collect_partial_from_holder's
# recusal rule) -- every holder contributes a partial to every opening
# attempt, same as the demo notebook's partials_from() helper.
from crypto.elgamal_threshold import compute_partial
from fl.trace import OpeningRequest, run_opening_protocol, revoke, residual_anonymity_bits

traceability_log = []       # one dict per flagged pseudonym: round, opened, H(T), etc.
_opening_ring = list(ring)  # REPORTING-ONLY shrinking ring, for the H(T) curve -- does NOT
                             # feed back into this run's own ring-signature verification (that
                             # would need §6.4 committee/ring reconfiguration, out of scope here;
                             # see FIX.md's traceability-wiring entries for why).
_n_opened_so_far = 0


def on_pseudonym_flagged(pseudonym_bytes, t_round, evidence_total, escrow_ciphertext, epoch, certificate):
    global _opening_ring, _n_opened_so_far
    request = OpeningRequest(
        epoch=epoch, accused_pseudonym=pseudonym_bytes, evidence_total=evidence_total,
        calibration_certificate=certificate, escrow_ciphertext=escrow_ciphertext,
    )
    transcript = f"opening:{epoch}:{pseudonym_bytes.hex()}".encode()
    partials = {
        i: compute_partial(escrow_ciphertext, key_holder_shares[i], pk_shares[i], transcript)
        for i in key_holder_ids
    }
    verdict = run_opening_protocol(request, partials, pk_shares, t, admin_authorized=True)

    entry = {
        "round": t_round, "epoch": epoch, "pseudonym": pseudonym_bytes.hex(),
        "evidence_total": evidence_total, "theta": certificate.theta, "opened": verdict.opened,
    }
    if verdict.opened:
        _opening_ring, record = revoke(verdict, _opening_ring, epoch)
        _n_opened_so_far += 1
        entry["new_ring_size"] = record.new_ring_size
        entry["residual_anonymity_bits"] = residual_anonymity_bits(len(ring), _n_opened_so_far)
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... OPENED "
              f"(evidence={evidence_total:.3f} > theta={certificate.theta:.3f}), "
              f"ring {len(ring)}->{record.new_ring_size}, H(T)={entry['residual_anonymity_bits']:.3f} bits")
    else:
        entry["reason"] = verdict.reason
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... flagged but "
              f"NOT opened ({verdict.reason})")
    traceability_log.append(entry)


In [ ]:
# 7. Build the model.
# ~20.6K params, same model as the already-committed MNIST baseline grid -- only the dataset (writer-disjoint non-IID FEMNIST) differs, per brutal-review gap #4's own scoping.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=62, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability via git (not Drive)

**Why a separate clone, a separate branch, and a callback -- not just
`git commit` inside `/content/FL-ZKP`:**
- A SEPARATE clone (`/content/FL-ZKP-checkpoints`) so committing a
  checkpoint can never touch the code checkout this notebook is actually
  importing from -- no risk of a `git checkout`/`reset` on the
  checkpoint side disturbing `sys.path`-imported modules mid-run.
- A dedicated branch (shown as `CKPT_BRANCH` in the cell below), never
  `main` -- small binary commits (one per round, re-run disconnects
  add more) have no business sitting in the same history a reviewer
  sees when they clone this repo to read the paper's actual code.
  **Each of the 4 MVP run notebooks gets its OWN branch** (FIX.md F17,
  2026-11-17) -- an earlier version shared one branch across all four,
  keyed apart only by checkpoint filename, and every single push
  collided when they ran concurrently (observed live: 12/12 rounds
  failed), even with a fetch+rebase retry (F16). A dedicated ref per
  notebook removes the race at the git level instead of mitigating it.
- `fl/round_loop.py`'s `on_checkpoint_saved` hook (2026-11-16) keeps
  the FL library itself git-agnostic -- it just calls back with
  `(checkpoint_path, round_idx)` after every local save; all the git
  mechanics below live here, in the notebook, not in the library.

**You'll need a GitHub Personal Access Token** with push access to
`YashasweeNarayan/FL-ZKP` (classic PAT, `repo` scope, or a fine-grained
token scoped to just this repo). The cell below prompts for it via
`getpass` (never echoed to the cell's output, never written to a file
or committed) and deletes it from memory immediately after use.

**On a disconnect and re-run**: this clone lives on local Colab disk,
so it does NOT survive the disconnect (same as the code clone) -- but
everything pushed to this notebook's own `CKPT_BRANCH` before the
disconnect DID survive, on GitHub. Re-running this cell re-clones
fresh and fetches that branch if it already exists, so the run cell's
`resume_from` still finds the latest checkpoint and continues from
there, not round 0.


In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints-baselinegrid-femnist-0"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case2-tiny_cnn-baselinegrid-signflip-10pct"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# Baseline-attack grid cell (FEMNIST): Sign-flip @ 10% malicious
# (5 of 50 clients), 20 rounds, §6.2
# traceability wired (opening handler cell above). Same
# cheater_delta_transform hook as the MNIST grid -- only the dataset
# differs here.
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from fl.models import unflatten_to_state_dict
from attacks.baseline_norm_attacks import (
    craft_sign_flip, craft_gaussian_noise, craft_scale_to_threshold, craft_low_and_slow,
)

REAL_ROUNDS = 20
CHEATER_CLIENT_IDX = list(range(5))

CKPT_PATH = f'{CKPT_DIR}/case2_tiny_cnn_baselinegrid_signflip_10pct_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed="bootstrap-tiny_cnn-case2-baselinegrid-signflip_10pct".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

# Deterministic per-cell RNG for the attack's own randomness -- same
# offset scheme as the MNIST grid, shifted by +500 so no (attack,
# fraction) cell's seed ever collides between the two grids even
# though both notebooks use the same CFG['seed'] base.
_attack_rng = np.random.default_rng(CFG['seed'] + 1500)
cheater_delta_transform = lambda d, bt: craft_sign_flip(d)

print("="*70)
print(f"[femnist-signflip_10pct] MALICIOUS CLIENTS: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%), attack='Sign-flip'")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"[femnist-signflip_10pct] found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

# FEMNIST's headline split (unseen_test) is a FemnistClientDataset
# dataclass (.images/.labels), NOT a plain tuple -- same unpack pattern
# _build_mvp_run_notebooks.py::run_cell uses for every other FEMNIST
# notebook in this repo.
_eval_ds = case2_data['unseen_test']
_eval_images, _eval_labels = _eval_ds.images, _eval_ds.labels


def _eval_fn(round_idx, w_t_now):
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}


final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case2_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_delta_transform=cheater_delta_transform,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
    on_pseudonym_flagged=on_pseudonym_flagged,
)
release_gpu_memory()

n_aborted = sum(1 for h in real_history if h.get('aborted'))
n_completed = len(real_history) - n_aborted
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"[femnist-signflip_10pct]   round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"[femnist-signflip_10pct]   round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

print(f"\n[femnist-signflip_10pct] {n_completed} of {len(real_history)} rounds completed, {n_aborted} aborted.")
if attacker_ever_accepted:
    print(f"[femnist-signflip_10pct] V3 FAIL: attacker(s) {sorted(attacker_ever_accepted)} had a crafted update accepted at least once.")
elif attacker_ever_rejected:
    print(f"[femnist-signflip_10pct] V3 PASS: every eligible attacker ({sorted(attacker_ever_rejected)}) was rejected by V3 every time it submitted.")
else:
    print(f"[femnist-signflip_10pct] INCONCLUSIVE: no attacker was ever sampled/eligible in any round.")

print(f"\n[femnist-signflip_10pct] §6 traceability summary:")
if traceability_log:
    for e in traceability_log:
        if e["opened"]:
            print(f"[femnist-signflip_10pct]   round {e['round']}: pseudonym {e['pseudonym'][:12]}... OPENED "
                  f"(evidence={e['evidence_total']:.3f} > theta={e['theta']:.3f}), "
                  f"H(T)={e['residual_anonymity_bits']:.3f} bits")
        else:
            print(f"[femnist-signflip_10pct]   round {e['round']}: pseudonym {e['pseudonym'][:12]}... flagged, NOT opened "
                  f"({e['reason']})")
    print(f"[femnist-signflip_10pct]   {_n_opened_so_far} identity(ies) opened across {len(traceability_log)} flagged pseudonym(s) this cell.")
else:
    print(f"[femnist-signflip_10pct]   no pseudonym ever crossed theta in these {REAL_ROUNDS} rounds.")

real_history_signflip_10pct = list(real_history)
n_aborted_signflip_10pct = n_aborted
n_completed_signflip_10pct = n_completed
attacker_ever_accepted_signflip_10pct = dict(attacker_ever_accepted)
attacker_ever_rejected_signflip_10pct = dict(attacker_ever_rejected)
traceability_log_signflip_10pct = list(traceability_log)
n_opened_signflip_10pct = _n_opened_so_far


## Grid cell 2 of 12 (FEMNIST, part 1 of 5): Sign-flip @ 20% malicious (10 of 50 clients)

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 6.5. §6.2 real opening-protocol handler, wired to
# run_training_loop's on_pseudonym_flagged callback (fl/round_loop.py,
# 2026-11-19). This is the SAME run_opening_protocol +
# revoke + residual_anonymity_bits machinery already validated
# standalone in notebooks/traceability_demo/TRACEABILITY_DEMO.ipynb
# (both against the local pure-Python backend and, per FIX.md, the
# real rbcl backend on Colab) -- nothing new is introduced here, this
# cell just triggers it from REAL per-round evidence instead of a
# hand-built test case, using the SAME committee (key_holder_shares,
# pk_shares) the ring cell above just built for escrow.
#
# None of these key holders is ever one of this run's clients (two
# disjoint simulated roles), so no holder ever recuses as
# the accused party (fl/trace.py::collect_partial_from_holder's
# recusal rule) -- every holder contributes a partial to every opening
# attempt, same as the demo notebook's partials_from() helper.
from crypto.elgamal_threshold import compute_partial
from fl.trace import OpeningRequest, run_opening_protocol, revoke, residual_anonymity_bits

traceability_log = []       # one dict per flagged pseudonym: round, opened, H(T), etc.
_opening_ring = list(ring)  # REPORTING-ONLY shrinking ring, for the H(T) curve -- does NOT
                             # feed back into this run's own ring-signature verification (that
                             # would need §6.4 committee/ring reconfiguration, out of scope here;
                             # see FIX.md's traceability-wiring entries for why).
_n_opened_so_far = 0


def on_pseudonym_flagged(pseudonym_bytes, t_round, evidence_total, escrow_ciphertext, epoch, certificate):
    global _opening_ring, _n_opened_so_far
    request = OpeningRequest(
        epoch=epoch, accused_pseudonym=pseudonym_bytes, evidence_total=evidence_total,
        calibration_certificate=certificate, escrow_ciphertext=escrow_ciphertext,
    )
    transcript = f"opening:{epoch}:{pseudonym_bytes.hex()}".encode()
    partials = {
        i: compute_partial(escrow_ciphertext, key_holder_shares[i], pk_shares[i], transcript)
        for i in key_holder_ids
    }
    verdict = run_opening_protocol(request, partials, pk_shares, t, admin_authorized=True)

    entry = {
        "round": t_round, "epoch": epoch, "pseudonym": pseudonym_bytes.hex(),
        "evidence_total": evidence_total, "theta": certificate.theta, "opened": verdict.opened,
    }
    if verdict.opened:
        _opening_ring, record = revoke(verdict, _opening_ring, epoch)
        _n_opened_so_far += 1
        entry["new_ring_size"] = record.new_ring_size
        entry["residual_anonymity_bits"] = residual_anonymity_bits(len(ring), _n_opened_so_far)
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... OPENED "
              f"(evidence={evidence_total:.3f} > theta={certificate.theta:.3f}), "
              f"ring {len(ring)}->{record.new_ring_size}, H(T)={entry['residual_anonymity_bits']:.3f} bits")
    else:
        entry["reason"] = verdict.reason
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... flagged but "
              f"NOT opened ({verdict.reason})")
    traceability_log.append(entry)


In [ ]:
# 7. Build the model.
# ~20.6K params, same model as the already-committed MNIST baseline grid -- only the dataset (writer-disjoint non-IID FEMNIST) differs, per brutal-review gap #4's own scoping.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=62, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability for this cell

Same git-backed checkpointing as the first cell in this part -- this cell gets its own clone directory and its own dedicated branch, so it can never collide with another cell's checkpoint, whether in this part, any of the other 4, or the MNIST grid.

In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints-baselinegrid-femnist-1"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case2-tiny_cnn-baselinegrid-signflip-20pct"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# Baseline-attack grid cell (FEMNIST): Sign-flip @ 20% malicious
# (10 of 50 clients), 20 rounds, §6.2
# traceability wired (opening handler cell above). Same
# cheater_delta_transform hook as the MNIST grid -- only the dataset
# differs here.
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from fl.models import unflatten_to_state_dict
from attacks.baseline_norm_attacks import (
    craft_sign_flip, craft_gaussian_noise, craft_scale_to_threshold, craft_low_and_slow,
)

REAL_ROUNDS = 20
CHEATER_CLIENT_IDX = list(range(10))

CKPT_PATH = f'{CKPT_DIR}/case2_tiny_cnn_baselinegrid_signflip_20pct_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed="bootstrap-tiny_cnn-case2-baselinegrid-signflip_20pct".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

# Deterministic per-cell RNG for the attack's own randomness -- same
# offset scheme as the MNIST grid, shifted by +500 so no (attack,
# fraction) cell's seed ever collides between the two grids even
# though both notebooks use the same CFG['seed'] base.
_attack_rng = np.random.default_rng(CFG['seed'] + 1501)
cheater_delta_transform = lambda d, bt: craft_sign_flip(d)

print("="*70)
print(f"[femnist-signflip_20pct] MALICIOUS CLIENTS: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%), attack='Sign-flip'")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"[femnist-signflip_20pct] found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

# FEMNIST's headline split (unseen_test) is a FemnistClientDataset
# dataclass (.images/.labels), NOT a plain tuple -- same unpack pattern
# _build_mvp_run_notebooks.py::run_cell uses for every other FEMNIST
# notebook in this repo.
_eval_ds = case2_data['unseen_test']
_eval_images, _eval_labels = _eval_ds.images, _eval_ds.labels


def _eval_fn(round_idx, w_t_now):
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}


final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case2_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_delta_transform=cheater_delta_transform,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
    on_pseudonym_flagged=on_pseudonym_flagged,
)
release_gpu_memory()

n_aborted = sum(1 for h in real_history if h.get('aborted'))
n_completed = len(real_history) - n_aborted
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"[femnist-signflip_20pct]   round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"[femnist-signflip_20pct]   round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

print(f"\n[femnist-signflip_20pct] {n_completed} of {len(real_history)} rounds completed, {n_aborted} aborted.")
if attacker_ever_accepted:
    print(f"[femnist-signflip_20pct] V3 FAIL: attacker(s) {sorted(attacker_ever_accepted)} had a crafted update accepted at least once.")
elif attacker_ever_rejected:
    print(f"[femnist-signflip_20pct] V3 PASS: every eligible attacker ({sorted(attacker_ever_rejected)}) was rejected by V3 every time it submitted.")
else:
    print(f"[femnist-signflip_20pct] INCONCLUSIVE: no attacker was ever sampled/eligible in any round.")

print(f"\n[femnist-signflip_20pct] §6 traceability summary:")
if traceability_log:
    for e in traceability_log:
        if e["opened"]:
            print(f"[femnist-signflip_20pct]   round {e['round']}: pseudonym {e['pseudonym'][:12]}... OPENED "
                  f"(evidence={e['evidence_total']:.3f} > theta={e['theta']:.3f}), "
                  f"H(T)={e['residual_anonymity_bits']:.3f} bits")
        else:
            print(f"[femnist-signflip_20pct]   round {e['round']}: pseudonym {e['pseudonym'][:12]}... flagged, NOT opened "
                  f"({e['reason']})")
    print(f"[femnist-signflip_20pct]   {_n_opened_so_far} identity(ies) opened across {len(traceability_log)} flagged pseudonym(s) this cell.")
else:
    print(f"[femnist-signflip_20pct]   no pseudonym ever crossed theta in these {REAL_ROUNDS} rounds.")

real_history_signflip_20pct = list(real_history)
n_aborted_signflip_20pct = n_aborted
n_completed_signflip_20pct = n_completed
attacker_ever_accepted_signflip_20pct = dict(attacker_ever_accepted)
attacker_ever_rejected_signflip_20pct = dict(attacker_ever_rejected)
traceability_log_signflip_20pct = list(traceability_log)
n_opened_signflip_20pct = _n_opened_so_far


## Grid cell 3 of 12 (FEMNIST, part 1 of 5): Sign-flip @ 30% malicious (15 of 50 clients)

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 6.5. §6.2 real opening-protocol handler, wired to
# run_training_loop's on_pseudonym_flagged callback (fl/round_loop.py,
# 2026-11-19). This is the SAME run_opening_protocol +
# revoke + residual_anonymity_bits machinery already validated
# standalone in notebooks/traceability_demo/TRACEABILITY_DEMO.ipynb
# (both against the local pure-Python backend and, per FIX.md, the
# real rbcl backend on Colab) -- nothing new is introduced here, this
# cell just triggers it from REAL per-round evidence instead of a
# hand-built test case, using the SAME committee (key_holder_shares,
# pk_shares) the ring cell above just built for escrow.
#
# None of these key holders is ever one of this run's clients (two
# disjoint simulated roles), so no holder ever recuses as
# the accused party (fl/trace.py::collect_partial_from_holder's
# recusal rule) -- every holder contributes a partial to every opening
# attempt, same as the demo notebook's partials_from() helper.
from crypto.elgamal_threshold import compute_partial
from fl.trace import OpeningRequest, run_opening_protocol, revoke, residual_anonymity_bits

traceability_log = []       # one dict per flagged pseudonym: round, opened, H(T), etc.
_opening_ring = list(ring)  # REPORTING-ONLY shrinking ring, for the H(T) curve -- does NOT
                             # feed back into this run's own ring-signature verification (that
                             # would need §6.4 committee/ring reconfiguration, out of scope here;
                             # see FIX.md's traceability-wiring entries for why).
_n_opened_so_far = 0


def on_pseudonym_flagged(pseudonym_bytes, t_round, evidence_total, escrow_ciphertext, epoch, certificate):
    global _opening_ring, _n_opened_so_far
    request = OpeningRequest(
        epoch=epoch, accused_pseudonym=pseudonym_bytes, evidence_total=evidence_total,
        calibration_certificate=certificate, escrow_ciphertext=escrow_ciphertext,
    )
    transcript = f"opening:{epoch}:{pseudonym_bytes.hex()}".encode()
    partials = {
        i: compute_partial(escrow_ciphertext, key_holder_shares[i], pk_shares[i], transcript)
        for i in key_holder_ids
    }
    verdict = run_opening_protocol(request, partials, pk_shares, t, admin_authorized=True)

    entry = {
        "round": t_round, "epoch": epoch, "pseudonym": pseudonym_bytes.hex(),
        "evidence_total": evidence_total, "theta": certificate.theta, "opened": verdict.opened,
    }
    if verdict.opened:
        _opening_ring, record = revoke(verdict, _opening_ring, epoch)
        _n_opened_so_far += 1
        entry["new_ring_size"] = record.new_ring_size
        entry["residual_anonymity_bits"] = residual_anonymity_bits(len(ring), _n_opened_so_far)
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... OPENED "
              f"(evidence={evidence_total:.3f} > theta={certificate.theta:.3f}), "
              f"ring {len(ring)}->{record.new_ring_size}, H(T)={entry['residual_anonymity_bits']:.3f} bits")
    else:
        entry["reason"] = verdict.reason
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... flagged but "
              f"NOT opened ({verdict.reason})")
    traceability_log.append(entry)


In [ ]:
# 7. Build the model.
# ~20.6K params, same model as the already-committed MNIST baseline grid -- only the dataset (writer-disjoint non-IID FEMNIST) differs, per brutal-review gap #4's own scoping.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=62, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability for this cell

Same git-backed checkpointing as the first cell in this part -- this cell gets its own clone directory and its own dedicated branch, so it can never collide with another cell's checkpoint, whether in this part, any of the other 4, or the MNIST grid.

In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints-baselinegrid-femnist-2"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case2-tiny_cnn-baselinegrid-signflip-30pct"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# Baseline-attack grid cell (FEMNIST): Sign-flip @ 30% malicious
# (15 of 50 clients), 20 rounds, §6.2
# traceability wired (opening handler cell above). Same
# cheater_delta_transform hook as the MNIST grid -- only the dataset
# differs here.
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from fl.models import unflatten_to_state_dict
from attacks.baseline_norm_attacks import (
    craft_sign_flip, craft_gaussian_noise, craft_scale_to_threshold, craft_low_and_slow,
)

REAL_ROUNDS = 20
CHEATER_CLIENT_IDX = list(range(15))

CKPT_PATH = f'{CKPT_DIR}/case2_tiny_cnn_baselinegrid_signflip_30pct_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed="bootstrap-tiny_cnn-case2-baselinegrid-signflip_30pct".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

# Deterministic per-cell RNG for the attack's own randomness -- same
# offset scheme as the MNIST grid, shifted by +500 so no (attack,
# fraction) cell's seed ever collides between the two grids even
# though both notebooks use the same CFG['seed'] base.
_attack_rng = np.random.default_rng(CFG['seed'] + 1502)
cheater_delta_transform = lambda d, bt: craft_sign_flip(d)

print("="*70)
print(f"[femnist-signflip_30pct] MALICIOUS CLIENTS: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%), attack='Sign-flip'")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"[femnist-signflip_30pct] found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

# FEMNIST's headline split (unseen_test) is a FemnistClientDataset
# dataclass (.images/.labels), NOT a plain tuple -- same unpack pattern
# _build_mvp_run_notebooks.py::run_cell uses for every other FEMNIST
# notebook in this repo.
_eval_ds = case2_data['unseen_test']
_eval_images, _eval_labels = _eval_ds.images, _eval_ds.labels


def _eval_fn(round_idx, w_t_now):
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}


final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case2_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_delta_transform=cheater_delta_transform,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
    on_pseudonym_flagged=on_pseudonym_flagged,
)
release_gpu_memory()

n_aborted = sum(1 for h in real_history if h.get('aborted'))
n_completed = len(real_history) - n_aborted
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"[femnist-signflip_30pct]   round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"[femnist-signflip_30pct]   round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

print(f"\n[femnist-signflip_30pct] {n_completed} of {len(real_history)} rounds completed, {n_aborted} aborted.")
if attacker_ever_accepted:
    print(f"[femnist-signflip_30pct] V3 FAIL: attacker(s) {sorted(attacker_ever_accepted)} had a crafted update accepted at least once.")
elif attacker_ever_rejected:
    print(f"[femnist-signflip_30pct] V3 PASS: every eligible attacker ({sorted(attacker_ever_rejected)}) was rejected by V3 every time it submitted.")
else:
    print(f"[femnist-signflip_30pct] INCONCLUSIVE: no attacker was ever sampled/eligible in any round.")

print(f"\n[femnist-signflip_30pct] §6 traceability summary:")
if traceability_log:
    for e in traceability_log:
        if e["opened"]:
            print(f"[femnist-signflip_30pct]   round {e['round']}: pseudonym {e['pseudonym'][:12]}... OPENED "
                  f"(evidence={e['evidence_total']:.3f} > theta={e['theta']:.3f}), "
                  f"H(T)={e['residual_anonymity_bits']:.3f} bits")
        else:
            print(f"[femnist-signflip_30pct]   round {e['round']}: pseudonym {e['pseudonym'][:12]}... flagged, NOT opened "
                  f"({e['reason']})")
    print(f"[femnist-signflip_30pct]   {_n_opened_so_far} identity(ies) opened across {len(traceability_log)} flagged pseudonym(s) this cell.")
else:
    print(f"[femnist-signflip_30pct]   no pseudonym ever crossed theta in these {REAL_ROUNDS} rounds.")

real_history_signflip_30pct = list(real_history)
n_aborted_signflip_30pct = n_aborted
n_completed_signflip_30pct = n_completed
attacker_ever_accepted_signflip_30pct = dict(attacker_ever_accepted)
attacker_ever_rejected_signflip_30pct = dict(attacker_ever_rejected)
traceability_log_signflip_30pct = list(traceability_log)
n_opened_signflip_30pct = _n_opened_so_far


In [ ]:
print("\n" + "="*70)
print("BASELINE-ATTACK GRID SUMMARY, FEMNIST (part, 3 of 12 cells) -- V3 + \u00a76 traceability")
print("="*70)
print(f"{'attack':>20} | {'fraction':>8} | {'rounds':>6} | {'aborted':>7} | {'V3 rejected':>11} | {'V3 accepted':>11} | {'traced':>6} | {'opened':>6}")
print(f"{'Sign-flip':>20} | {'10%':>8} | {len(real_history_signflip_10pct):>6} | {n_aborted_signflip_10pct:>7} | {len(attacker_ever_rejected_signflip_10pct):>11} | {len(attacker_ever_accepted_signflip_10pct):>11} | {len(traceability_log_signflip_10pct):>6} | {n_opened_signflip_10pct:>6}")
print(f"{'Sign-flip':>20} | {'20%':>8} | {len(real_history_signflip_20pct):>6} | {n_aborted_signflip_20pct:>7} | {len(attacker_ever_rejected_signflip_20pct):>11} | {len(attacker_ever_accepted_signflip_20pct):>11} | {len(traceability_log_signflip_20pct):>6} | {n_opened_signflip_20pct:>6}")
print(f"{'Sign-flip':>20} | {'30%':>8} | {len(real_history_signflip_30pct):>6} | {n_aborted_signflip_30pct:>7} | {len(attacker_ever_rejected_signflip_30pct):>11} | {len(attacker_ever_accepted_signflip_30pct):>11} | {len(traceability_log_signflip_30pct):>6} | {n_opened_signflip_30pct:>6}")
print("\n(This is one of 5 parts, FEMNIST grid -- combine with the other parts' printed tables for the full 12-cell picture; nothing here auto-merges across separate notebook sessions.)")
